# Lesson 10 — SGD, Momentum, Adam & AdamW From Scratch

## 学习目标

上一节已经完成：

$$
Token\ IDs
\rightarrow
Logits
\rightarrow
CrossEntropy
\rightarrow
Loss
\rightarrow
Backward
$$

调用：

`loss.backward()`

之后，每个可训练参数都会得到对应的梯度。

但梯度本身不会自动修改模型参数。

这一节要解决的问题是：

> 已经知道 Gradient 以后，应该怎样更新 Parameter？

完成本节后，应能够：

1. 理解 Parameter、Gradient、Optimizer 三者的关系；
2. 从 Gradient Descent 推导 SGD；
3. 手动完成一次 Parameter Update；
4. 理解为什么参数更新必须放在 `torch.no_grad()` 中；
5. 理解为什么每个 Training Step 都要清空 Gradient；
6. 理解 Learning Rate 的作用；
7. 理解 Momentum 为什么能够加速训练；
8. 理解 Momentum Buffer；
9. 理解 Adam 的 First Moment；
10. 理解 Adam 的 Second Moment；
11. 理解 Adam Bias Correction；
12. 从零实现 Adam；
13. 理解 L2 Regularization 与 Weight Decay；
14. 理解 Adam 和 AdamW 的关键区别；
15. 从零实现 AdamW；
16. 与 `torch.optim.AdamW` 做参数更新 Reference Test；
17. 理解 Optimizer State 的内存成本；
18. 为 CS336 Assignment 1 的 AdamW 实现做好准备。


## 1. Gradient Descent

假设模型只有一个参数：

$$
\theta
$$

Loss 为：

$$
L(\theta)
$$

经过 Backward，我们得到：

$$
\nabla_\theta L
=
\frac{\partial L}{\partial \theta}
$$

Gradient 告诉我们：

> 参数向哪个方向移动，会让 Loss 增大最快。

因此，为了让 Loss 下降，我们沿 Gradient 的反方向更新：

$$
\boxed{
\theta
\leftarrow
\theta
-
\eta\nabla_\theta L
}
$$

其中：

$$
\eta
$$

是 Learning Rate。

可以把整个过程理解为：

$$
Parameter
$$

$$
\downarrow Forward
$$

$$
Loss
$$

$$
\downarrow Backward
$$

$$
Gradient
$$

$$
\downarrow Optimizer
$$

$$
Updated\ Parameter
$$


In [1]:
import torch

theta = torch.tensor(3.0, requires_grad=True)

loss = theta**2
loss.backward()

print("theta before:", theta.item())
print("gradient:", theta.grad.item())

theta before: 3.0
gradient: 6.0


## 2. Manual SGD Update

假设：

$$
\theta=3
$$

梯度：

$$
\nabla_\theta L=6
$$

Learning Rate：

$$
\eta=0.1
$$

那么：

$$
\theta_{new}
=
3
-
0.1\times6
$$

因此：

$$
\theta_{new}
=
2.4
$$

这就是最基本的 SGD Parameter Update。


In [2]:
learning_rate = 0.1

with torch.no_grad():
    theta -= learning_rate * theta.grad

print("theta after:", theta.item())

theta after: 2.4000000953674316


## 3. 为什么 Parameter Update 使用 `torch.no_grad()`？

参数更新本身不是模型 Forward Computation 的一部分。

我们希望 Autograd 记录：

$$
Parameter
\rightarrow
Model
\rightarrow
Loss
$$

但不希望继续记录：

$$
Parameter
\rightarrow
Optimizer\ Update
\rightarrow
Parameter
\rightarrow
Optimizer\ Update
\rightarrow\dots
$$

因此 Parameter Update 通常写在：

`torch.no_grad()`

环境中。

例如：

`theta -= learning_rate * theta.grad`

这个操作应该修改 Parameter 的值，

但不应该建立新的 Computational Graph。

所以训练循环中经常看到：

```text
Forward
↓
Loss
↓
Backward
↓
no_grad()
↓
Parameter Update
```

## 4. Gradient Accumulation

PyTorch 默认会累积 Gradient。

也就是说：

第一次：

`loss.backward()`

之后：

$$
grad=g_1
$$

如果不清零，再次：

`loss.backward()`

则：

$$
grad=g_1+g_2
$$

而不是：

$$
grad=g_2
$$

这在 Gradient Accumulation 中非常有用。

但普通 Training Loop 每一步通常只希望使用当前 Batch 的 Gradient。

所以通常需要：

`optimizer.zero_grad()`

或者手动：

`parameter.grad = None`


In [3]:
x = torch.tensor(2.0, requires_grad=True)

loss = x**2
loss.backward()

print("after first backward:", x.grad.item())

loss = x**2
loss.backward()

print("after second backward:", x.grad.item())

after first backward: 4.0
after second backward: 8.0


In [4]:
x.grad = None

loss = x**2
loss.backward()

print("after clearing:", x.grad.item())

after clearing: 4.0


## 5. 最小 Optimization Loop

现在已经可以组成最简单的 Training Loop：

### Step 1

清空旧 Gradient。

### Step 2

Forward。

### Step 3

计算 Loss。

### Step 4

Backward。

### Step 5

根据 Gradient 更新 Parameter。

因此：

```text
for each step:

    zero gradient

        ↓

    forward

        ↓

    loss

        ↓

    backward

        ↓

    parameter update
```

In [5]:
theta = torch.tensor(5.0, requires_grad=True)

learning_rate = 0.1

for step in range(10):
    theta.grad = None

    loss = theta**2
    loss.backward()

    with torch.no_grad():
        theta -= learning_rate * theta.grad

    print(f"step={step:02d}", f"theta={theta.item():.6f}", f"loss={loss.item():.6f}")


step=00 theta=4.000000 loss=25.000000
step=01 theta=3.200000 loss=16.000000
step=02 theta=2.560000 loss=10.240001
step=03 theta=2.048000 loss=6.553600
step=04 theta=1.638400 loss=4.194304
step=05 theta=1.310720 loss=2.684354
step=06 theta=1.048576 loss=1.717986
step=07 theta=0.838861 loss=1.099511
step=08 theta=0.671089 loss=0.703687
step=09 theta=0.536871 loss=0.450360


## 6. Learning Rate

SGD：

$$
\theta_{t+1}
=
\theta_t
-
\eta g_t
$$

其中：

$$
g_t
=
\nabla_\theta L_t
$$

Learning Rate：

$$
\eta
$$

控制一次更新走多远。

如果 $\eta$ 太小：

$$
Parameter
$$

每次移动很少，

训练可能非常慢。

如果 $\eta$ 太大：

可能：

$$
Overshoot
$$

甚至：

$$
Divergence
$$

因此 Learning Rate 是训练中最重要的 Hyperparameter 之一。


In [6]:
def optimize_quadratic(learning_rate: float, steps: int = 10) -> list[float]:
    theta = torch.tensor(5.0, requires_grad=True)

    history = []

    for _ in range(steps):
        theta.grad = None

        loss = theta**2
        loss.backward()

        with torch.no_grad():
            theta -= learning_rate * theta.grad

        history.append(theta.item())

    return history


for lr in [0.01, 0.1, 0.9, 1.1]:
    print("lr:", lr, optimize_quadratic(lr))


lr: 0.01 [4.900000095367432, 4.802000045776367, 4.705960273742676, 4.611841201782227, 4.519604206085205, 4.4292120933532715, 4.340627670288086, 4.253815174102783, 4.168738842010498, 4.085363864898682]
lr: 0.1 [4.0, 3.200000047683716, 2.559999942779541, 2.047999858856201, 1.6383998394012451, 1.3107198476791382, 1.0485758781433105, 0.8388606905937195, 0.6710885763168335, 0.5368708372116089]
lr: 0.9 [-4.0, 3.1999998092651367, -2.559999465942383, 2.047999382019043, -1.638399362564087, 1.3107194900512695, -1.0485756397247314, 0.8388605117797852, -0.6710883378982544, 0.5368705987930298]
lr: 1.1 [-6.0, 7.200000762939453, -8.64000129699707, 10.368001937866211, -12.44160270690918, 14.929924011230469, -17.915908813476562, 21.49909210205078, -25.798912048339844, 30.958694458007812]


# 7. Stochastic Gradient Descent

完整 Dataset 的 Gradient：

$$
\nabla_\theta
\frac{1}{N}
\sum_{i=1}^{N}
L_i
$$

如果每次都使用全部 Dataset，

叫做：

Full-Batch Gradient Descent。

现实中的神经网络通常使用一个 Mini-Batch：

$$
B
\ll
N
$$

估计 Gradient：

$$
g_t
=
\frac{1}{B}
\sum_{i\in Batch_t}
\nabla_\theta L_i
$$

然后：

$$
\theta_{t+1}
=
\theta_t-\eta g_t
$$

因为 Mini-Batch 会随机采样，

所以 Gradient 带有噪声。

这就是：

Stochastic Gradient Descent

或者更准确地说：

Mini-Batch SGD。


## 8. Optimizer 的核心职责

Optimizer 本质上做两件事情：

### 1. 保存需要更新的 Parameter

例如：

$$
W
$$

$$
b
$$

### 2. 根据 Parameter Gradient 修改 Parameter

最简单的 SGD：

$$
p
\leftarrow
p-\eta p.grad
$$

因此 Optimizer 并不负责：

- Forward；
- Loss；
- Backward。

它只负责：

> 使用已经计算好的 Gradient 更新 Parameter。


In [7]:
from __future__ import annotations

from collections.abc import Iterable


class SGDFromScratch:
    def __init__(self, params: Iterable[torch.Tensor], lr: float) -> None:
        if lr < 0:
            raise ValueError("lr must be non-negative")

        self.params = list(params)
        self.lr = lr

    @torch.no_grad()
    def step(self) -> None:
        for param in self.params:
            if param.grad is None:
                continue

            param -= self.lr * param.grad

    def zero_grad(self) -> None:
        for param in self.params:
            param.grad = None


In [8]:
parameter = torch.tensor(3.0, requires_grad=True)
optimizer = SGDFromScratch([parameter], lr=0.1)

for step in range(5):
    optimizer.zero_grad()

    loss = parameter**2
    loss.backward()

    optimizer.step()

    print(
        f"step={step}", f"parameter={parameter.item():.6f}", f"loss={loss.item():.6f}"
    )


step=0 parameter=2.400000 loss=9.000000
step=1 parameter=1.920000 loss=5.760000
step=2 parameter=1.536000 loss=3.686400
step=3 parameter=1.228800 loss=2.359296
step=4 parameter=0.983040 loss=1.509950


## 10. Reference Testing

从现在开始，实现数学 Primitive 时采用统一测试思想：

> 自己的实现，与可信 Reference Implementation 对齐。

这比单纯判断：

"看起来 Loss 在下降"

严格得多。

对于 Optimizer，我们应该比较：

- 相同初始 Parameter；
- 相同 Gradient；
- 相同 Hyperparameter；

经过一步 Update 后：

$$
Parameter_{ours}
\approx
Parameter_{reference}
$$

后面实现 AdamW 时也使用完全相同的方法。


In [9]:
torch.manual_seed(42)

parameter_ours = torch.randn(4, requires_grad=True)
parameter_reference = parameter_ours.detach().clone().requires_grad_(True)

gradient = torch.randn(4)

parameter_ours.grad = gradient.clone()
parameter_reference.grad = gradient.clone()

optimizer_ours = SGDFromScratch([parameter_ours], lr=0.01)
optimizer_reference = torch.optim.SGD([parameter_reference], lr=0.01)

optimizer_ours.step()
optimizer_reference.step()

print("ours:", parameter_ours)
print("reference:", parameter_reference)
print("same:", torch.allclose(parameter_ours, parameter_reference))

ours: tensor([0.3479, 0.1307, 0.2124, 0.2367], requires_grad=True)
reference: tensor([0.3479, 0.1307, 0.2124, 0.2367], requires_grad=True)
same: True


# 11. 为什么需要 Momentum？

考虑二维 Loss Surface。

一个方向非常陡：

$$
y
$$

另一个方向比较平缓：

$$
x
$$

SGD 可能出现：

```text
↘
  ↙
    ↘
      ↙
        ↘


## 12. Momentum

一种常见写法：

$$
v_t
=
\beta v_{t-1}
+
g_t
$$

然后：

$$
\theta_{t+1}
=
\theta_t
-
\eta v_t
$$

其中：

$$
v_t
$$

称为：
Momentum Buffer
或 Velocity。

$\beta$ 通常接近：

$$
0.9
$$

因此当前 Update Direction 同时包含：

- 当前 Gradient；
- 历史 Gradient。

如果多个 Step 的 Gradient 都指向相似方向，这些 Gradient 会累积。

如果 Gradient 在某个方向不断正负震荡，则会部分抵消。


In [10]:
gradients = torch.tensor([1.0, 1.0, 1.0, 1.0])

beta = 0.9

velocity = torch.tensor(0.0)

for gradient in gradients:
    velocity = beta * velocity + gradient

    print("gradient:", gradient.item(), "velocity:", velocity.item())


gradient: 1.0 velocity: 1.0
gradient: 1.0 velocity: 1.899999976158142
gradient: 1.0 velocity: 2.7100000381469727
gradient: 1.0 velocity: 3.438999891281128


## 13. Exponential Moving Average

Momentum 与后面的 Adam 都大量使用：

Exponential Moving Average

基本形式：

$$
m_t
=
\beta m_{t-1}
+
(1-\beta)x_t
$$

展开：

$$
m_t
=
(1-\beta)x_t
+
\beta(1-\beta)x_{t-1}
+
\beta^2(1-\beta)x_{t-2}
+\cdots
$$

因此越久以前的数据权重越小。

当：

$$
\beta=0.9
$$

意味着模型对最近一段历史保留较强记忆。

可以粗略理解为：

$$
Effective\ Window
\approx
\frac{1}{1-\beta}
$$

所以：

$$
\beta=0.9
$$

大约对应：

$$
10
$$

个 Step 的时间尺度。

而：

$$
\beta=0.999
$$

大约对应：

$$
1000
$$

个 Step。


# 14. Adam 的核心思想

SGD 使用：

$$
g_t
$$

直接更新参数。

Momentum 开始记录：

$$
Gradient\ Direction
$$

Adam 再进一步：

> 同时记录 Gradient 的一阶矩和二阶矩。

First Moment：

$$
m_t
$$

追踪 Gradient 的平均方向。

Second Moment：

$$
v_t
$$

追踪 Gradient 平方的平均大小。

Adam：
Adaptive Moment Estimation

因此名字来自：
Adaptive
+
Moment。


## 15. First Moment

Adam 定义：

$$
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t
$$

通常：

$$
\beta_1=0.9
$$

这是 Gradient 的 Exponential Moving Average。

可以把它理解为：

> 当前比较稳定的 Gradient Direction。


## 16. Second Moment

第二个状态：

$$
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2
$$

通常：

$$
\beta_2=0.999
$$

注意：

$$
g_t^2
$$

是逐元素平方。

所以：

$$
v_t
$$

始终非负。

它衡量：

> 每个 Parameter Dimension 上，Gradient 通常有多大。

如果某个方向 Gradient 长期很大：

$$
v_t
$$

也会较大。

Adam 会让这个方向的实际 Step 相对变小。

因此 Adam 可以对不同 Parameter Dimension 使用不同的 Effective Learning Rate。


## 17. Adam 的 Update Intuition

忽略 Bias Correction 后：

$$
\Delta\theta
\approx
-\eta
\frac{m_t}
{\sqrt{v_t}+\epsilon}
$$

其中：

$$
m_t
$$

决定：
方向。

而：

$$
\sqrt{v_t}
$$

控制：
尺度。

如果某个维度 Gradient 很大：

$$
v_t\uparrow
$$

因此：

$$
\frac{1}{\sqrt{v_t}}
\downarrow
$$

实际 Update 会被缩小。

如果某个维度 Gradient 较小：
实际 Update 相对更大。

这就是 Adam 中：
Adaptive Learning Rate
的来源之一。


# 18. Bias Correction

Adam 初始化：

$$
m_0=0
$$

$$
v_0=0
$$

考虑第一个 Step：

$$
m_1
=
(1-\beta_1)g_1
$$

如果：

$$
\beta_1=0.9
$$

那么：

$$
m_1
=
0.1g_1
$$

明显偏小。

原因是：
EMA 是从 0 开始初始化的。
在训练早期，
$m_t$ 和 $v_t$ 都会产生：
Initialization Bias。

因此 Adam 使用 Bias Correction。


## 19. First Moment Bias Correction

定义：

$$
\hat{m}_t
=
\frac{m_t}
{1-\beta_1^t}
$$

第一个 Step：

$$
m_1
=
(1-\beta_1)g_1
$$

因此：

$$
\hat{m}_1
=
\frac{
(1-\beta_1)g_1
}{
1-\beta_1
}
$$

所以：

$$
\hat{m}_1
=
g_1
$$

Bias Correction 修正了 EMA 初始为零造成的偏小。


## 20. Second Moment Bias Correction

类似地：

$$
\hat{v}_t
=
\frac{v_t}
{1-\beta_2^t}
$$

于是 Adam 使用：
$
\hat m_t
$
和：
$
\hat v_t
$

而不是原始：
$
m_t
$
和：
$
v_t
$


# 21. Adam Algorithm

对于 Step：

$$
t=1,2,\dots
$$

Gradient：

$$
g_t
=
\nabla_\theta L_t
$$

### First Moment

$$
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t
$$

### Second Moment

$$
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2
$$

### Bias Correction

$$
\hat m_t
=
\frac{m_t}
{1-\beta_1^t}
$$

$$
\hat v_t
=
\frac{v_t}
{1-\beta_2^t}
$$

### Parameter Update

$$
\boxed{
\theta_t
=
\theta_{t-1}
-
\eta
\frac{
\hat m_t
}{
\sqrt{\hat v_t}
+
\epsilon
}
}
$$

常见默认值：

$$
\beta_1=0.9
$$

$$
\beta_2=0.999
$$

$$
\epsilon=10^{-8}
$$


## 22. 手工计算第一个 Adam Step

假设：

$$
g_1=2
$$

$$
\beta_1=0.9
$$

$$
\beta_2=0.999
$$

First Moment：

$$
m_1
=
0.9(0)
+
0.1(2)
=
0.2
$$

Second Moment：

$$
v_1
=
0.999(0)
+
0.001(2^2)
$$

$$
v_1
=
0.004
$$

Bias Correction：

$$
\hat m_1
=
\frac{0.2}{1-0.9}
=
2
$$

$$
\hat v_1
=
\frac{0.004}{1-0.999}
=
4
$$

所以：

$$
\frac{\hat m_1}
{\sqrt{\hat v_1}}
=
\frac{2}{2}
=
1
$$

忽略 $\epsilon$，

第一个 Adam Update 大约是：

$$
-\eta
$$

这也是理解 Adam Scale Invariance 的一个重要起点。


In [11]:
gradient = torch.tensor(2.0)

beta1 = 0.9
beta2 = 0.999

m = torch.tensor(0.0)
v = torch.tensor(0.0)

m = beta1 * m + (1 - beta1) * gradient
v = beta2 * v + (1 - beta2) * gradient**2

step = 1

m_hat = m / (1 - beta1**step)
v_hat = v / (1 - beta2**step)

print("m:", m)
print("v:", v)
print("m_hat:", m_hat)
print("v_hat:", v_hat)

m: tensor(0.2000)
v: tensor(0.0040)
m_hat: tensor(2.)
v_hat: tensor(4.)


In [ ]:
def adam_undate(
    parameter: torch.Tensor,
    gradient: torch.Tensor,
    fist_moment: torch.Tensor,
    second_moment: torch.Tensor,
    step: int,
    *,
    lr: float,
    beta1: float,
    beta2: float,
    eps: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Update the parameter using Adam optimizer.
    """
    first_moment = beta1 * fist_moment + (1 - beta1) * gradient
    second_moment = beta2 * second_moment + (1 - beta2) * gradient.square()

    first_moment_hat = first_moment / (1 - beta1**step)
    second_moment_hat = second_moment / (1 - beta2**step)

    parameter -= lr * first_moment_hat / (second_moment_hat.sqrt() + eps)

    return parameter, first_moment, second_moment

# 24. Optimizer State

SGD 对每个 Parameter 只需要：

$$
Parameter
$$

本身。

Momentum 还需要：

$$
Momentum\ Buffer
$$

Adam 则需要为每个 Parameter 保存：

$$
m_t
$$

和：

$$
v_t
$$

因此 Adam 的 Optimizer State 大约需要：

$$
2\times
Parameter\ Count
$$

个额外数值。

假设 Parameter 使用 FP32：

每个数：

$$
4\ bytes
$$

那么仅 Adam Moment State：

$$
8\ bytes
$$

per parameter。

如果还有：

- Parameter；
- Gradient；
- Master Weight；
- Optimizer State；

训练显存会远远大于模型 Parameter 本身。

这也是后面 Distributed Training 和 Optimizer State Sharding 非常重要的原因。


## 25. Optimizer State 与 Parameter Shape 相同

假设：

$$
W.shape=(D,H)
$$

Adam 需要：

$$
m_W.shape=(D,H)
$$

以及：

$$
v_W.shape=(D,H)
$$

如果：

$$
b.shape=(H,)
$$

则：

$$
m_b.shape=(H,)
$$

$$
v_b.shape=(H,)
$$

因此每个 Parameter 都有自己独立的 Optimizer State。

可以理解成：

```text
parameter_1
├── first_moment
└── second_moment

parameter_2
├── first_moment
└── second_moment

...
```

# 26. L2 Regularization

在理解 AdamW 之前，必须区分：
L2 Regularization
和：
Weight Decay。

L2 Regularization 把参数范数加入 Loss：

$$
L_{total}
=
L_{data}
+
\frac{\lambda}{2}
\|\theta\|_2^2
$$

对参数求导：

$$
\nabla_\theta L_{total}
=
\nabla_\theta L_{data}
+
\lambda\theta
$$

于是 Gradient 中多出：

$$
\lambda\theta
$$

对于普通 SGD：

$$
\theta
\leftarrow
\theta
-
\eta
(
g+\lambda\theta
)
$$

展开：

$$
\theta
\leftarrow
(1-\eta\lambda)\theta
-
\eta g
$$

这里出现了参数缩小：

$$
(1-\eta\lambda)\theta
$$

所以在标准 SGD 中，
L2 Regularization 和 Weight Decay 可以得到等价的更新形式。


## 27. Weight Decay

Weight Decay 更直接的思想是：

> 每个 Optimization Step 都让 Parameter 稍微缩小。

一种形式：

$$
\theta
\leftarrow
(1-\eta\lambda)\theta
$$

然后再执行 Gradient Update。

也可以写成：

$$
\theta
\leftarrow
\theta
-
\eta\lambda\theta
$$

因此 Parameter 会逐步向 0 收缩。

$\lambda$ 是 Weight Decay Coefficient。


# 28. Adam 中的关键区别

对于 SGD：

$$
g+\lambda\theta
$$

直接乘相同 Learning Rate。

所以 L2 Regularization 可以转化成 Weight Decay。

但是 Adam 会对 Gradient 做：

$$
\frac{
\hat m_t
}{
\sqrt{\hat v_t}+\epsilon
}
$$

这种 Parameter-Wise Adaptive Scaling。

如果把：

$$
\lambda\theta
$$

直接加入 Gradient：

$$
g'
=
g+\lambda\theta
$$

那么 Regularization Term 也会进入：

- First Moment；
- Second Moment；
- Adaptive Normalization。

因此它不再等价于：

> 单独把参数按固定比例缩小。

AdamW 的核心思想就是：

> 把 Weight Decay 从 Gradient Adaptation 中解耦。


# 29. AdamW

AdamW：
Adam
+
Decoupled Weight Decay。

核心更新可以理解为两部分。

### Adam Update

$$
\theta
\leftarrow
\theta
-
\eta
\frac{
\hat m_t
}{
\sqrt{\hat v_t}+\epsilon
}
$$

### Weight Decay

$$
\theta
\leftarrow
\theta
-
\eta\lambda\theta
$$

合并：

$$
\boxed{
\theta
\leftarrow
\theta
-
\eta
\left[
\frac{
\hat m_t
}{
\sqrt{\hat v_t}+\epsilon
}
+
\lambda\theta
\right]
}
$$

关键点：
$
\lambda\theta
$
不参与：
$
m_t
$
和：
$
v_t
$
的计算。

这就是：
Decoupled Weight Decay。


## 30. Adam 与 AdamW

### Adam + L2

先修改 Gradient：

$$
g
\leftarrow
g+\lambda\theta
$$

然后整个 Gradient 进入：
$
m_t
$
和：
$
v_t
$
计算。

---

### AdamW

Moment State 只使用真正的 Data Gradient：
$
g
$

Weight Decay 单独作用于 Parameter：

$$
\theta
\leftarrow
(1-\eta\lambda)\theta
$$


# 31. AdamW Step

给定 Gradient：$g_t$

### Step 1：First Moment

$$
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t
$$

### Step 2：Second Moment

$$
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2
$$

### Step 3：Bias Correction

$$
\hat m_t
=
\frac{m_t}
{1-\beta_1^t}
$$

$$
\hat v_t
=
\frac{v_t}
{1-\beta_2^t}
$$

### Step 4：Adam Update

$$
\theta
\leftarrow
\theta
-
\eta
\frac{
\hat m_t
}{
\sqrt{\hat v_t}+\epsilon
}
$$

### Step 5：Decoupled Weight Decay

$$
\theta
\leftarrow
\theta
-
\eta\lambda\theta
$$

实际实现中也可以把两部分组织成等价的单步形式。


In [13]:
from collections.abc import Iterable


class AdamWFromScratch:
    def __init__(
        self,
        params: Iterable[torch.Tensor],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.0,
    ) -> None:
        if lr < 0:
            raise ValueError("lr must be non-negative")

        beta1, beta2 = betas

        if not 0.0 <= beta1 < 1.0:
            raise ValueError("beta1 must be in [0, 1)")

        if not 0.0 <= beta2 < 1.0:
            raise ValueError("beta2 must be in [0, 1)")

        if eps < 0:
            raise ValueError("eps must be non-negative")

        if weight_decay < 0:
            raise ValueError("weight_decay must be non-negative")

        self.params = list(params)

        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay

        self.step_count = 0

        self.first_moments = [torch.zeros_like(param) for param in self.params]
        self.second_moments = [torch.zeros_like(param) for param in self.params]

    @torch.no_grad()
    def step(self) -> None:
        self.step_count += 1

        beta1 = self.beta1
        beta2 = self.beta2

        bias_correction1 = 1.0 - beta1**self.step_count
        bias_correction2 = 1.0 - beta2**self.step_count

        for param, first_moment, second_moment in zip(
            self.params, self.first_moments, self.second_moments, strict=True
        ):
            if param.grad is None:
                continue

            grad = param.grad

            first_moment.mul_(beta1).add_(grad, alpha=1.0 - beta1)
            second_moment.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

            first_moment_hat = first_moment / bias_correction1
            second_moment_hat = second_moment / bias_correction2

            if self.weight_decay != 0:
                param.mul_(1.0 - self.lr * self.weight_decay)

            param.addcdiv_(
                first_moment_hat,
                second_moment_hat.sqrt().add_(self.eps),
                value=-self.lr,
            )

    def zero_grad(self) -> None:
        for param in self.params:
            param.grad = None


## 33. In-Place Optimizer Update

上面的实现使用：

`mul_()`

`add_()`

`addcmul_()`

`addcdiv_()`

末尾的：

`_`

表示 In-Place Operation。

例如：

`x.mul_(0.9)`

相当于修改：

`x`

本身。

Optimizer State：

$$
m_t
$$

和：

$$
v_t
$$

每一步都会被更新。

因此没有必要每次创建新的 Moment Tensor。

这可以减少：

- 临时 Tensor；
- Memory Allocation；
- Memory Traffic。

不过：

> 学习阶段首先保证数学正确，再考虑 In-Place Optimization。

如果 In-Place 写法影响理解，可以先写最直接的 Reference Version。


In [14]:
@torch.no_grad()
def adamw_reference_step(
    parameter: torch.Tensor,
    gradient: torch.Tensor,
    first_moment: torch.Tensor,
    second_moment: torch.Tensor,
    step: int,
    *,
    lr: float,
    beta1: float,
    beta2: float,
    eps: float,
    weight_decay: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    first_moment = beta1 * first_moment + (1.0 - beta1) * gradient
    second_moment = beta2 * second_moment + (1.0 - beta2) * gradient.square()

    first_moment_hat = first_moment / (1.0 - beta1**step)
    second_moment_hat = second_moment / (1.0 - beta2**step)

    parameter = parameter * (1.0 - lr * weight_decay)
    parameter = parameter - lr * first_moment_hat / (second_moment_hat.sqrt() + eps)

    return (parameter, first_moment, second_moment)


In [15]:
torch.manual_seed(42)

parameter_ours = torch.randn(8, requires_grad=True)
parameter_reference = parameter_ours.detach().clone().requires_grad_(True)

gradient = torch.randn(8)

parameter_ours.grad = gradient.clone()
parameter_reference.grad = gradient.clone()

optimizer_ours = AdamWFromScratch(
    [parameter_ours], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
)
optimizer_reference = torch.optim.AdamW(
    [parameter_reference], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
)

optimizer_ours.step()
optimizer_reference.step()

print("ours:", parameter_ours)
print("reference:", parameter_reference)
print("maximum error:", (parameter_ours - parameter_reference).abs().max().item())

ours: tensor([ 0.3357,  0.1278,  0.2335,  0.2293, -1.1238, -0.1853,  2.2092, -0.6390],
       requires_grad=True)
reference: tensor([ 0.3357,  0.1278,  0.2335,  0.2293, -1.1238, -0.1853,  2.2092, -0.6390],
       requires_grad=True)
maximum error: 0.0


## 36. 为什么必须测试多个 Step？

只测试 AdamW 第一个 Step 不够。

Adam 的核心包含：
$
m_t
$
和：
$
v_t
$
也就是历史状态。

一个实现可能：

- 第一步正确；
- Moment 更新错误；
- 第二步以后开始偏离。

因此 Optimizer Test 应该至少覆盖：

$$
Multiple\ Steps
$$

并且每一步使用新的 Gradient。


In [16]:
torch.manual_seed(123)

parameter_ours = torch.randn(16, requires_grad=True)
parameter_reference = parameter_ours.detach().clone().requires_grad_(True)

optimizer_ours = AdamWFromScratch(
    [parameter_ours], lr=3e-4, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1
)
optimizer_reference = torch.optim.AdamW(
    [parameter_reference], lr=3e-4, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.1
)

for step in range(20):
    gradient = torch.randn_like(parameter_ours)

    parameter_ours.grad = gradient.clone()
    parameter_reference.grad = gradient.clone()

    optimizer_ours.step()
    optimizer_reference.step()

    assert torch.allclose(parameter_ours, parameter_reference, atol=1e-6, rtol=1e-5)

    optimizer_ours.zero_grad()
    optimizer_reference.zero_grad(set_to_none=True)

print("20-step AdamW reference test passed.")


20-step AdamW reference test passed.


In [17]:
torch.manual_seed(7)

parameters_ours = [
    torch.randn(3, 4, requires_grad=True),
    torch.randn(4, requires_grad=True),
]
parameters_reference = [
    (parameter.detach().clone().requires_grad_(True)) for parameter in parameters_ours
]

optimizer_ours = AdamWFromScratch(parameters_ours, lr=1e-3, weight_decay=0.01)
optimizer_reference = torch.optim.AdamW(
    parameters_reference, lr=1e-3, weight_decay=0.01
)

for _ in range(10):
    for ours, reference in zip(parameters_ours, parameters_reference, strict=True):
        gradient = torch.randn_like(ours)

        ours.grad = gradient.clone()
        reference.grad = gradient.clone()

    optimizer_ours.step()
    optimizer_reference.step()

    for ours, reference in zip(parameters_ours, parameters_reference, strict=True):
        assert torch.allclose(ours, reference, atol=1e-6, rtol=1e-5)

    optimizer_ours.zero_grad()
    optimizer_reference.zero_grad(set_to_none=True)

print("Multi-parameter test passed.")


Multi-parameter test passed.


# 38. `grad = None` vs `grad.zero_()`

清空 Gradient 有两种常见方式。

### 方法 1

`parameter.grad.zero_()`

把已有 Gradient Tensor 全部写成 0。

Gradient Tensor 仍然存在。

### 方法 2

`parameter.grad = None`

直接表示：

> 当前没有 Gradient。

现代 PyTorch 中常见：

`optimizer.zero_grad(set_to_none=True)`

使用 `None` 通常可以：

- 避免额外的 zero memory write；
- 更清晰地区分“没有 Gradient”和“Gradient 正好是 0”。

在我们的学习实现中：

`param.grad = None`

是一个很合理的默认选择。


# 39. Optimizer Memory Accounting

假设模型有：

$$
N
$$

个 Parameter。

如果全部使用 FP32：

### Parameters

$$
4N\ bytes
$$

### Gradients

$$
4N\ bytes
$$

### Adam First Moment

$$
4N\ bytes
$$

### Adam Second Moment

$$
4N\ bytes
$$

仅这些部分：

$$
16N\ bytes
$$

也就是：

$$
16\ bytes/parameter
$$

例如：

$$
1B
$$

Parameter：

$$
16\ GB
$$

还没有计算：

- Activations；
- Temporary Buffers；
- CUDA Context；
- Mixed Precision Master Weights；
- Communication Buffers。

这就是为什么大模型训练的 Memory Accounting 非常重要。


## 40. Training Memory != Parameter Memory

很多初学者会看到：

$$
1B\times4\ bytes
=
4GB
$$

于是认为：

> 1B FP32 Model 只需要 4GB GPU。

这是错误的。

训练至少还涉及：

$$
Parameters
$$

$$
+
$$

$$
Gradients
$$

$$
+
$$

$$
Optimizer\ States
$$

$$
+
$$

$$
Activations
$$

对于 Adam 类 Optimizer，

Optimizer State 本身就可能是 Parameter Memory 的两倍。

这也是之后 CS336 Systems 中会进一步研究：

- Mixed Precision；
- Gradient Checkpointing；
- ZeRO；
- FSDP；
- Optimizer State Sharding；

的原因。


# 41. Parameter Groups

真实 Transformer 训练中，通常不会简单地对所有 Parameter 使用完全相同的 Weight Decay。

常见策略之一是：对 Matrix Weight：$W$使用 Weight Decay。

而对某些：

- Bias；
- Normalization Scale；

不使用 Weight Decay。

因此 PyTorch Optimizer 支持：Parameter Groups。

概念上：

```text
group 1
├── parameters
├── lr
└── weight_decay = 0.1

group 2
├── bias / norm parameters
├── lr
└── weight_decay = 0
```

# 42. AdamW 中的 Decay Scale

AdamW 的 Decay：

$$
\theta
\leftarrow
(1-\eta\lambda)\theta
$$

注意其中同时出现：
$
\eta
$
和：
$
\lambda
$

所以每一步实际 Decay Scale 为：

$$
\eta\lambda
$$

例如：

$$
\eta=10^{-3}
$$

$$
\lambda=0.1
$$

则：

$$
\eta\lambda
=
10^{-4}
$$

单步：

$$
\theta
\rightarrow
0.9999\theta
$$

看起来很小。

但经过大量 Training Steps 后会逐渐累积。


# 43. Adam 中的 Epsilon

Adam Update：

$$
\frac{
\hat m_t
}{
\sqrt{\hat v_t}
+
\epsilon
}
$$

如果某个 Parameter Dimension：

$$
\hat v_t
\approx0
$$

那么：

$$
\sqrt{\hat v_t}
\approx0
$$

除法可能变得：

- 非常大；
- 数值不稳定；
- 甚至出现除零。

因此加入：

$$
\epsilon>0
$$

通常：

$$
10^{-8}
$$

作为 Numerical Stability Term。

这与上一节 Stable Softmax 中的思想类似：

> 数学公式正确还不够，工程实现必须考虑有限精度浮点数。


# 44. Adam / AdamW Hyperparameters

最主要的 Hyperparameter：

### Learning Rate

$$
\eta
$$

控制整体 Step Size。

### First Moment Decay

$$
\beta_1
$$

控制 Gradient Direction 的时间尺度。

典型：

$$
0.9
$$

### Second Moment Decay

$$
\beta_2
$$

控制 Gradient Magnitude 的时间尺度。

典型：

$$
0.999
$$

LLM Training 中也经常使用其它值，例如：

$$
0.95
$$

### Epsilon

$$
\epsilon
$$

用于 Numerical Stability。

### Weight Decay

$$
\lambda
$$

控制 Parameter Shrinkage。

因此 AdamW 不是：

> 一个完全不需要调参的 Optimizer。

Learning Rate、Betas、Weight Decay 都会影响训练动态。


# 45. Optimizer 的边界

AdamW 很强，

但它不能修复：

- 错误的 Loss；
- 错误的 Target；
- 错误的 Attention Mask；
- NaN Gradient；
- 错误的 Model Architecture；
- Data Bug。

Optimizer 只看到：

$$
Parameter
$$

和：

$$
Gradient
$$

如果 Gradient 本身错误，

AdamW 只会非常高效地沿错误方向更新。

因此 Training Debugging 的基本顺序通常仍然是：

$$
Correctness
$$

$$
\downarrow
$$

$$
Numerical\ Stability
$$

$$
\downarrow
$$

$$
Optimization
$$

$$
\downarrow
$$

$$
Performance
$$


# 46. 完整 Optimization Pipeline

现在已经可以第一次完整描述一个 Training Step。

### Forward

$$
X
\rightarrow
Model
\rightarrow
Logits
$$

### Loss

$$
Logits
+
Targets
\rightarrow
CrossEntropy
$$

得到：

$$
L
$$

### Backward

`loss.backward()`

得到每个 Parameter：

$$
\nabla_\theta L
$$

### Optimizer

AdamW 根据：

$$
g_t
$$

更新：

$$
m_t
$$

$$
v_t
$$

并得到：

$$
\theta_{t+1}
$$

### Zero Grad

清空：

$$
Gradient
$$

然后进入下一个 Batch。

完整循环：

```text
zero_grad
   ↓
forward
   ↓
loss
   ↓
backward
   ↓
optimizer.step
   ↓
next batch
```

In [18]:
torch.manual_seed(42)

x = torch.randn(128, 4)

true_weight = torch.tensor([[2.0], [-1.0], [0.5], [3.0]])
targets = x @ true_weight

weight = torch.randn(4, 1, requires_grad=True)
optimizer = AdamWFromScratch([weight], lr=0.05, weight_decay=0.0)


for step in range(200):
    optimizer.zero_grad()

    predictions = x @ weight

    loss = (predictions - targets).square().mean()
    loss.backward()

    optimizer.step()

    if step % 40 == 0:
        print(f"step={step:03d}", f"loss={loss.item():.8f}")


print("\nlearned weight:")
print(weight)
print("\ntrue weight:")
print(true_weight)

step=000 loss=15.09297657
step=040 loss=1.44086266
step=080 loss=0.04315004
step=120 loss=0.00000706
step=160 loss=0.00000864

learned weight:
tensor([[ 2.0000],
        [-1.0001],
        [ 0.5000],
        [ 2.9998]], requires_grad=True)

true weight:
tensor([[ 2.0000],
        [-1.0000],
        [ 0.5000],
        [ 3.0000]])


In [19]:
torch.manual_seed(2026)

x = torch.randn(64, 8)

targets = torch.randn(64, 4)

weight_ours = torch.randn(8, 4, requires_grad=True)
weight_reference = weight_ours.detach().clone().requires_grad_(True)

optimizer_ours = AdamWFromScratch(
    [weight_ours], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
)
optimizer_reference = torch.optim.AdamW(
    [weight_reference], lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01
)

for step in range(50):
    optimizer_ours.zero_grad()
    optimizer_reference.zero_grad(set_to_none=True)

    prediction_ours = x @ weight_ours
    prediction_reference = x @ weight_reference

    loss_ours = (prediction_ours - targets).square().mean()
    loss_reference = (prediction_reference - targets).square().mean()

    loss_ours.backward()
    loss_reference.backward()

    optimizer_ours.step()
    optimizer_reference.step()

    assert torch.allclose(weight_ours, weight_reference, atol=1e-6, rtol=1e-5)


print("Training-loop AdamW reference test passed.")


Training-loop AdamW reference test passed.


# 49. 常见错误

## 错误 1：忘记清空 Gradient

如果每个 Training Step 都不清空：

$$
grad
$$

会不断累积。

---

## 错误 2：在 Autograd Graph 中直接更新 Parameter

Parameter Update 应该放在：

`torch.no_grad()`

或者 Optimizer 自己的 no-grad `step()` 中。

---

## 错误 3：Adam 忘记 Bias Correction

如果直接使用：

$$
m_t
$$

和：

$$
v_t
$$

训练早期会受到 Zero Initialization Bias。

---

## 错误 4：Second Moment 没有平方 Gradient

Adam Second Moment 使用：

$$
g_t^2
$$

不是：

$$
g_t
$$

---

## 错误 5：把 AdamW 当成 Adam + L2

AdamW 的关键：

> Weight Decay 不进入 Moment Estimation。

---

## 错误 6：所有 Parameter 共用一个 Moment Tensor

每个 Parameter 都应该有对应的：

$$
m
$$

和：

$$
v
$$

而且 Shape 与 Parameter 一致。

---

## 错误 7：Step Count 从 0 直接用于 Bias Correction

Bias Correction 中：

$$
1-\beta^t
$$

第一个实际 Update 应使用：

$$
t=1
$$

否则：

$$
1-\beta^0=0
$$

会发生除零。

---

## 错误 8：Reference Test 只测一步

Adam / AdamW 有状态。

必须测试：
Multiple Steps。

---

## 错误 9：认为 Loss 下降就说明 Optimizer 正确

错误实现也可能短时间让 Loss 下降。

更严格的验证方法是：

$$
Parameter_{ours}
\approx
Parameter_{PyTorch}
$$
